# 02 – SQL exploration of the event log

First look at the BPI Challenge 2019 log with DuckDB (SQL on the Parquet file): size, activities, durations, transitions, and data-quality problems found along the way.

In [1]:
import duckdb

con = duckdb.connect()
con.execute("CREATE VIEW events AS SELECT * FROM read_parquet('../data/bpi2019.parquet')")


## 1. Overview
1.6 million events, 251,734 orders, 42 activities, 1,975 vendors.

In [2]:
con.sql("""
SELECT COUNT(*) AS events,
       COUNT(DISTINCT "case:concept:name") AS cases,
       COUNT(DISTINCT "concept:name") AS activities,
       COUNT(DISTINCT "case:Vendor") AS vendors,
       MIN("time:timestamp") AS first_event,
       MAX("time:timestamp") AS last_event
FROM events
""").df()


,events,cases,activities,vendors,first_event,last_event
0,1595923,251734,42,1975,1948-01-26 23:59:00+01:00,2020-04-09 23:59:00+02:00


## 2. Most frequent activities

In [3]:
con.sql("""
SELECT "concept:name" AS activity,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM events
GROUP BY 1
ORDER BY n DESC
LIMIT 15
""").df()

,activity,n,pct
0,Record Goods Receipt,314097,19.68
1,Create Purchase Order Item,251734,15.77
2,Record Invoice Receipt,228760,14.33
3,Vendor creates invoice,219919,13.78
4,Clear Invoice,194393,12.18
5,Record Service Entry Sheet,164975,10.34
6,Remove Payment Block,57136,3.58
7,Create Purchase Requisition Item,46592,2.92
8,Receive Order Confirmation,32065,2.01
9,Change Quantity,21449,1.34


## 3. Order durations (raw data)
The maximum of 25,671 days (about 70 years) is impossible for a purchase order. It points to wrong dates in the data.

In [4]:
con.sql("""
WITH c AS (
  SELECT "case:concept:name" AS case_id,
         COUNT(*) AS n_events,
         MIN("time:timestamp") AS start_ts,
         MAX("time:timestamp") AS end_ts
  FROM events
  GROUP BY 1
)
SELECT COUNT(*) AS cases,
       ROUND(AVG(n_events), 2) AS avg_events_per_case,
       MEDIAN(date_diff('day', start_ts, end_ts)) AS median_days,
       quantile_cont(date_diff('day', start_ts, end_ts), 0.9) AS p90_days,
       MAX(date_diff('day', start_ts, end_ts)) AS max_days
FROM c
""").df()


,cases,avg_events_per_case,median_days,p90_days,max_days
0,251734,6.34,64.0,126.0,25671


## 4. Transitions between activities (raw data)
Average time between consecutive activities. The averages are pulled up by a few extreme orders, e.g. *Vendor creates invoice → Create Purchase Order Item* averages 108.8 days but has a median of 12.0 days. Medians and p90 are used from now on.

In [5]:
con.sql("""
WITH ordered AS (
  SELECT "case:concept:name" AS case_id,
         "concept:name" AS activity,
         "time:timestamp" AS ts,
         LAG("concept:name") OVER (PARTITION BY "case:concept:name" ORDER BY "time:timestamp") AS prev_activity,
         LAG("time:timestamp") OVER (PARTITION BY "case:concept:name" ORDER BY "time:timestamp") AS prev_ts
  FROM events
)
SELECT prev_activity, activity,
       COUNT(*) AS n,
       ROUND(AVG(date_diff('second', prev_ts, ts)) / 86400.0, 2) AS avg_days,
       ROUND(MEDIAN(date_diff('second', prev_ts, ts)) / 86400.0, 2) AS median_days
FROM ordered
WHERE prev_activity IS NOT NULL
GROUP BY 1, 2
HAVING COUNT(*) >= 1000
ORDER BY avg_days DESC
LIMIT 15
""").df()

,prev_activity,activity,n,avg_days,median_days
0,Vendor creates invoice,Create Purchase Order Item,3319,108.81,11.98
1,Clear Invoice,Cancel Invoice Receipt,2111,84.45,85.14
2,Record Invoice Receipt,Clear Invoice,134955,43.72,36.06
3,Remove Payment Block,Clear Invoice,49424,36.33,25.08
4,Clear Invoice,Record Invoice Receipt,3958,31.37,3.75
5,Vendor creates invoice,Clear Invoice,1443,29.76,9.37
6,Record Invoice Receipt,Cancel Invoice Receipt,2697,29.72,11.04
7,Record Goods Receipt,Record Invoice Receipt,112182,21.64,12.80
8,Record Invoice Receipt,Remove Payment Block,37621,21.35,8.04
9,Vendor creates debit memo,Record Invoice Receipt,1317,17.94,9.46


## 5. Data quality: wrong dates
Events dated 1948, 1993, 2001, 2008 … while the real log covers 2018.

In [6]:
con.sql("""
SELECT year("time:timestamp") AS yr,
       COUNT(*) AS n_events,
       COUNT(DISTINCT "case:concept:name") AS n_cases
FROM events
GROUP BY 1
ORDER BY 1
""").df()

,yr,n_events,n_cases
0,1948,10,5
1,1993,9,9
2,2001,22,17
3,2008,45,45
4,2015,3,2
5,2016,6,3
6,2017,223,185
7,2018,1550468,251532
8,2019,45135,32680
9,2020,2,2


In [7]:
con.sql("""
SELECT "concept:name" AS activity,
       COUNT(*) AS n,
       MIN("time:timestamp") AS earliest
FROM events
WHERE "time:timestamp" < TIMESTAMPTZ '2017-01-01'
GROUP BY 1
ORDER BY n DESC
""").df()

,activity,n,earliest
0,Vendor creates invoice,77,1948-01-26 23:59:00+01:00
1,Vendor creates debit memo,18,1948-01-26 23:59:00+01:00


### Effect on the analysis
80 orders have an event before 2017. They are excluded. The median and p90 duration do not change (64 and 126 days), but the maximum drops from 25,671 to 819 days.

In [8]:
con.sql("""
WITH c AS (
  SELECT "case:concept:name" AS case_id,
         MIN("time:timestamp") AS start_ts,
         MAX("time:timestamp") AS end_ts
  FROM events GROUP BY 1
)
SELECT
  COUNT(*) AS cases,
  COUNT(*) FILTER (WHERE start_ts < TIMESTAMPTZ '2017-01-01') AS cases_pre_2017,
  MEDIAN(date_diff('day', start_ts, end_ts)) AS median_all,
  MEDIAN(date_diff('day', start_ts, end_ts)) FILTER (WHERE start_ts >= TIMESTAMPTZ '2017-01-01') AS median_clean,
  quantile_cont(date_diff('day', start_ts, end_ts), 0.9) FILTER (WHERE start_ts >= TIMESTAMPTZ '2017-01-01') AS p90_clean,
  MAX(date_diff('day', start_ts, end_ts)) FILTER (WHERE start_ts >= TIMESTAMPTZ '2017-01-01') AS max_clean
FROM c
""").df()

,cases,cases_pre_2017,median_all,median_clean,p90_clean,max_clean
0,251734,80,64.0,64.0,126.0,819


## 6. Clean view
Orders with any event before 2017 are removed: 1,593,429 events, 251,654 orders.

In [9]:
con.execute("""
CREATE OR REPLACE VIEW events_clean AS
SELECT * FROM events
WHERE "case:concept:name" NOT IN (
    SELECT DISTINCT "case:concept:name"
    FROM events
    WHERE "time:timestamp" < TIMESTAMPTZ '2017-01-01'
)
""")
con.sql("""
SELECT COUNT(*) AS events,
       COUNT(DISTINCT "case:concept:name") AS cases
FROM events_clean
""").df()

,events,cases
0,1593429,251654


## 7. Slowest transitions (clean data)
Ranked by count × median days, i.e. where the most waiting time accumulates. Median and 90th percentile instead of averages.

Note: here events are ordered by timestamp only. Notebooks 03 and 04 also use the row position in the file to order events of the same day, so counts differ by one or two in a few rows (e.g. 112,115 here, 112,114 in notebook 04).

In [10]:
con.sql("""
WITH ordered AS (
  SELECT "case:concept:name" AS case_id,
         "concept:name" AS activity,
         "time:timestamp" AS ts,
         LAG("concept:name") OVER (PARTITION BY "case:concept:name" ORDER BY "time:timestamp") AS prev_activity,
         LAG("time:timestamp") OVER (PARTITION BY "case:concept:name" ORDER BY "time:timestamp") AS prev_ts
  FROM events_clean
)
SELECT prev_activity, activity,
       COUNT(*) AS n,
       ROUND(MEDIAN(date_diff('second', prev_ts, ts)) / 86400.0, 1) AS median_days,
       ROUND(quantile_cont(date_diff('second', prev_ts, ts), 0.9) / 86400.0, 1) AS p90_days
FROM ordered
WHERE prev_activity IS NOT NULL
GROUP BY 1, 2
HAVING COUNT(*) >= 1000
ORDER BY n * median_days DESC
LIMIT 15
""").df()

,prev_activity,activity,n,median_days,p90_days
0,Record Invoice Receipt,Clear Invoice,134816,36.1,91.0
1,Record Goods Receipt,Record Invoice Receipt,112115,12.8,48.1
2,Remove Payment Block,Clear Invoice,49419,25.3,92.3
3,Create Purchase Order Item,Record Goods Receipt,77151,12.8,32.2
4,Create Purchase Order Item,Vendor creates invoice,103142,6.5,28.3
5,Vendor creates invoice,Record Invoice Receipt,95171,4.7,35.5
6,Record Invoice Receipt,Remove Payment Block,37616,8.0,61.3
7,Clear Invoice,Cancel Invoice Receipt,2079,86.6,168.1
8,Vendor creates invoice,Record Goods Receipt,110131,1.4,5.3
9,Receive Order Confirmation,Record Goods Receipt,20398,5.1,18.7
